# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal Check 1: Staleness (days_since_last_update)

I checked whether older, un-updated content is more likely to be declining. The answer is mixed. In my two biggest groups of pages — content updated in the last 30 days, and content untouched for 90–180 days — older content does decline more often (51% vs 61%), which supports the idea. But once content passes 180 days without an update, the pattern falls apart, and I barely have any pages that old to check (some groups had as few as 5 pages), so I can't trust that part of the data.

Verdict: MIXED

Signal Check 2: CTR-vs-Position (avg_position)

I checked whether pages ranked lower in search results are more likely to be declining. For most of the range, this held up — as position gets worse (from top-3 down to "striking distance," around position 10–20), decline rate does climb (50% → 61%), which is what I expected. But for pages ranked very deep (past position 50), the decline rate actually drops instead of climbing further, which goes against the assumption.

Verdict: MIXED

In plain terms: both signals are directionally useful — they tell a real, believable story for most of your content — but neither one is a clean, one-way relationship across the entire range. That's honest and useful to know before building a rule on top of them, rather than assuming they'd behave perfectly.

My Rule

A page is worth flagging for review if it has gone a while without an update AND its average search position has slipped past the top 10. Both signals point toward declining performance, and flagging pages where both are true should catch the clearest cases first.

Reason codes:

stale_and_slipping — both signals triggered (highest priority)
stale_only — content is old, but ranking is still fine
slipping_only — ranking has dropped, but content is still fairly fresh
neither — no flags triggered

In [13]:
staleness_bins = pd.cut(df["days_since_last_update"], bins=[0, 30, 90, 180, 365, 10000])
staleness_table = df.groupby(staleness_bins)["is_declining_label"].agg(["mean", "count"])
staleness_table

,mean,count
days_since_last_update,,
"(0, 30]",0.511377,20480
"(30, 90]",0.588571,175
"(90, 180]",0.611057,9171
"(180, 365]",0.467456,169
"(365, 10000]",0.600000,5


In [14]:
import pandas as pd

df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
print(df.shape)
df.head()

(30000, 52)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
position_bins = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 50, 1000])
position_table = df.groupby(position_bins)["is_declining_label"].agg(["mean", "count"])
position_table


,mean,count
avg_position,,
"(0, 3]",0.497809,1141
"(3, 10]",0.569414,11842
"(10, 20]",0.609515,7273
"(20, 50]",0.561799,7225
"(50, 1000]",0.343227,1314


In [16]:
# Signal 1: staleness flag
df["is_stale"] = (df["days_since_last_update"] >= 90).astype(int)

# Signal 2: slipping position flag (worse than top 10, and has real position data)
df["is_slipping"] = ((df["avg_position"] > 10) & (df["avg_position"] > 0)).astype(int)

# Reason code based on which signals triggered
def get_reason_code(row):
    if row["is_stale"] == 1 and row["is_slipping"] == 1:
        return "stale_and_slipping"
    elif row["is_stale"] == 1:
        return "stale_only"
    elif row["is_slipping"] == 1:
        return "slipping_only"
    else:
        return "neither"

df["reason_code"] = df.apply(get_reason_code, axis=1)

# Score: both signals triggered = highest priority
df["score"] = df["is_stale"] + df["is_slipping"]

# Action label
df["action"] = df["reason_code"].apply(lambda r: "refresh" if r != "neither" else "no_action")

# Build the ranked queue
queue = df[["content_id", "score", "reason_code", "action", "days_since_last_update", "avg_position", "is_declining_label"]]
queue = queue.sort_values("score", ascending=False)

# Write to CSV
import os
os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

print(queue.shape)
queue.head(10)

(30000, 7)


,content_id,score,reason_code,action,days_since_last_update,avg_position,is_declining_label
13,content_a5a2fbc76336,2,stale_and_slipping,refresh,103,39.8,0
14,content_91067a14431a,2,stale_and_slipping,refresh,104,27.1,1
19563,content_55d970081c8a,2,stale_and_slipping,refresh,92,49.0,0
19567,content_5ef037068d32,2,stale_and_slipping,refresh,104,14.1,0
19551,content_989adc1cd30b,2,stale_and_slipping,refresh,104,11.1,1
29954,content_5daabd0e31f3,2,stale_and_slipping,refresh,104,25.8,1
29956,content_20decd85a0c2,2,stale_and_slipping,refresh,104,31.0,1
19574,content_8c7447df63f5,2,stale_and_slipping,refresh,104,13.8,0
19579,content_47c84cf19b5d,2,stale_and_slipping,refresh,104,23.9,1
19580,content_c8ce24d44cb9,2,stale_and_slipping,refresh,104,32.7,0


In [17]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
p_at_50 = precision_at_k(df["score"], df["is_declining_label"], 50)
p_at_100 = precision_at_k(df["score"], df["is_declining_label"], 100)

print(f"Base rate (overall decline rate): {base_rate:.3f}")
print(f"Precision@50: {p_at_50:.3f}")
print(f"Precision@100: {p_at_100:.3f}")

Base rate (overall decline rate): 0.542
Precision@50: 0.620
Precision@100: 0.630


## 3. Top-20 review

Top-20 Review

All 20 top-ranked pages triggered stale_and_slipping (both signals fired: 90+ days since update, position past top-10) and were labeled refresh. Since the rule's score only goes up to 2, everything with both signals ties at the top — a real limitation of a simple additive score, and worth flagging honestly rather than pretending these 20 are finely ranked against each other.

content_a5a2fbc76336 — refresh, stale (103d) + slipping (pos 39.8). Wrong if this page is evergreen and holding steady traffic despite age (label shows not declining).
content_91067a14431a — refresh, stale (104d) + slipping (pos 27.1). Correctly flagged (label shows declining).
content_55d970081c8a — refresh, stale (92d) + slipping (pos 49.0). Wrong if it's a stable long-tail page rather than a true decline.
content_5ef037068d32 — refresh, stale (104d) + slipping (pos 14.1). Wrong if position 14 is "good enough" and traffic hasn't actually dropped.
content_989adc1cd30b — refresh, stale (104d) + slipping (pos 11.1). Correctly flagged.
content_5daabd0e31f3 — refresh, stale (104d) + slipping (pos 25.8). Correctly flagged.
content_20decd85a0c2 — refresh, stale (104d) + slipping (pos 31.0). Correctly flagged.
content_8c7447df63f5 — refresh, stale (104d) + slipping (pos 13.8). Wrong if this page's traffic has stayed flat rather than dropping.
content_47c84cf19b5d — refresh, stale (104d) + slipping (pos 23.9). Correctly flagged.
content_c8ce24d44cb9 — refresh, stale (104d) + slipping (pos 32.7). Wrong if it's a low-priority page that never had much traffic to lose.
content_113ebfccd893 — refresh, stale (104d) + slipping (pos 14.8). Correctly flagged.
content_278982959883 — refresh, stale (104d) + slipping (pos 67.4). Wrong if a page ranked this deep never had meaningful visibility to decline from.
content_be12f5f1f683 — refresh, stale (104d) + slipping (pos 47.0). Wrong if traffic has been flat, not falling.
content_020a0bf78894 — refresh, stale (104d) + slipping (pos 11.3). Correctly flagged.
content_db13a3ac2d38 — refresh, stale (104d) + slipping (pos 17.8). Correctly flagged.
content_bf57a6284e77 — refresh, stale (104d) + slipping (pos 24.0). Correctly flagged.
content_72c8c4a76ef1 — refresh, stale (98d) + slipping (pos 11.6). Wrong if this page is holding steady rather than declining.
content_4fe0d777d87c — refresh, stale (104d) + slipping (pos 13.4). Correctly flagged.
content_f7031183d2d0 — refresh, stale (104d) + slipping (pos 20.4). Correctly flagged.
content_a519180b7a3f — refresh, stale (104d) + slipping (pos 16.6). Wrong if this page's ranking has been consistently in this spot for a while, not newly slipping.

Pattern observed: roughly half of the top 20 (rows 1, 3, 4, 8, 10, 12, 13, 17, 20 — 9 of 20) are labeled as NOT actually declining, even though both signals fired. This lines up with the ~62% precision@50 we measured earlier — the rule catches real cases more often than chance, but is still wrong on a meaningful chunk of its highest-confidence picks.

In [18]:
top20 = queue.head(20)
top20


,content_id,score,reason_code,action,days_since_last_update,avg_position,is_declining_label
13,content_a5a2fbc76336,2,stale_and_slipping,refresh,103,39.8,0
14,content_91067a14431a,2,stale_and_slipping,refresh,104,27.1,1
19563,content_55d970081c8a,2,stale_and_slipping,refresh,92,49.0,0
19567,content_5ef037068d32,2,stale_and_slipping,refresh,104,14.1,0
19551,content_989adc1cd30b,2,stale_and_slipping,refresh,104,11.1,1
29954,content_5daabd0e31f3,2,stale_and_slipping,refresh,104,25.8,1
29956,content_20decd85a0c2,2,stale_and_slipping,refresh,104,31.0,1
19574,content_8c7447df63f5,2,stale_and_slipping,refresh,104,13.8,0
19579,content_47c84cf19b5d,2,stale_and_slipping,refresh,104,23.9,1
19580,content_c8ce24d44cb9,2,stale_and_slipping,refresh,104,32.7,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
# Leakage check: confirm we did NOT use trend_direction, trend_pct, or any future-window column
rule_inputs = ["days_since_last_update", "avg_position"]
forbidden = ["trend_direction", "trend_pct", "is_declining_label", 
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]

leaked = [col for col in rule_inputs if col in forbidden]
print("Columns used in the rule:", rule_inputs)
print("Any leaked/forbidden columns used?", leaked if leaked else "None — clean")



Columns used in the rule: ['days_since_last_update', 'avg_position']
Any leaked/forbidden columns used? None — clean


Weak Picks Summary

Of the top 20 ranked pages, 9 were flagged stale_and_slipping but were not actually declining (is_declining_label = 0): rows for content_a5a2fbc76336, content_55d970081c8a, content_5ef037068d32, content_8c7447df63f5, content_c8ce24d44cb9, content_278982959883, content_be12f5f1f683, content_72c8c4a76ef1, and content_a519180b7a3f.

The common thread: these are pages where both signals technically fired (stale + slipping), but the page's actual traffic hasn't dropped — likely evergreen or low-traffic pages that were never going to lose much regardless of position or update recency. This is the core weakness of a two-signal additive rule: it can't distinguish "old and low-ranked but stable" from "old and low-ranked and actively declining." A future model could likely improve on this by incorporating trend/volatility signals the rule deliberately excludes (to avoid leakage).

Leakage check: confirmed clean — the rule only uses days_since_last_update and avg_position, no label-derived or future-window columns

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.